# Sprocket Central Customer Analytics Project

This notebook analyses the Sprocket Central practice dataset. The goal is to identify useful customer and transaction patterns that could support marketing decisions, customer targeting, and business reporting.

**Main business questions**

1. How reliable is the transaction/order process?
2. Which brands and product sizes are most popular?
3. Which states show the strongest activity among new customers?
4. How are existing customers distributed by gender and wealth segment?

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Allow this notebook to import the reusable project code in ../src
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from kpmg_sprocket_analysis import (
    load_workbook_sheets,
    clean_transactions,
    clean_new_customers,
    clean_customer_demographic,
    clean_customer_address,
    build_summary_tables,
    save_summary_tables,
    create_visualisations,
)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Mentor_KPMG_Sprocket_Project_Dataset.xlsx"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

## 1. Load the Excel workbook

The source file contains four main sheets used for analysis: transactions, new customers, customer demographics, and customer addresses.

In [2]:
sheets = load_workbook_sheets(DATA_PATH)

for name, df in sheets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

transactions: 20,000 rows × 13 columns
new_customers: 1,000 rows × 23 columns
customer_demographic: 4,000 rows × 13 columns
customer_address: 3,999 rows × 6 columns


## 2. Quick data-quality check

Before analysing the data, I check missing values and obvious inconsistent categories. This helps show the cleaning decisions transparently.

In [3]:
missing_summary = []

for name, df in sheets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    for column, missing_count in missing.items():
        missing_summary.append({
            "table": name,
            "column": column,
            "missing_count": int(missing_count),
            "missing_percentage": round(missing_count / len(df) * 100, 2),
        })

pd.DataFrame(missing_summary)

,table,column,missing_count,missing_percentage
0,transactions,online_order,360,1.80
1,transactions,brand,197,0.98
2,transactions,product_line,197,0.98
3,transactions,product_class,197,0.98
4,transactions,product_size,197,0.98
5,transactions,standard_cost,197,0.98
6,transactions,product_first_sold_date,197,0.98
7,new_customers,job_industry_category,165,16.50
8,new_customers,job_title,106,10.60
9,new_customers,last_name,29,2.90


## 3. Clean the data

Cleaning steps applied:

- Standardised gender values such as `F`, `M`, and `Femal`.
- Standardised state names such as `New South Wales` to `NSW`.
- Created a transaction-level `profit` field from `list_price - standard_cost`.
- Renamed unnamed score columns in the new customer list.
- Dropped the inconsistent `default` column from the demographic table because it is not analytically useful.

In [4]:
transactions = clean_transactions(sheets["transactions"])
new_customers = clean_new_customers(sheets["new_customers"])
customer_demographic = clean_customer_demographic(sheets["customer_demographic"])
customer_address = clean_customer_address(sheets["customer_address"])

transactions.head()

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date,profit
0,1,2,2950,2017-02-25,Offline,Approved,Solex,Standard,medium,medium,71.49,53.62,41245.0,17.87
1,2,3,3120,2017-05-21,Online,Approved,Trek Bicycles,Standard,medium,large,2091.47,388.92,41701.0,1702.55
2,3,37,402,2017-10-16,Offline,Approved,OHM Cycles,Standard,low,medium,1793.43,248.82,36361.0,1544.61
3,4,88,3135,2017-08-31,Offline,Approved,Norco Bicycles,Standard,medium,medium,1198.46,381.10,36145.0,817.36
4,5,78,787,2017-10-01,Online,Approved,Giant Bicycles,Standard,medium,large,1765.30,709.48,42226.0,1055.82


## 4. Build summary tables

In [5]:
summary_tables = build_summary_tables(transactions, new_customers, customer_demographic)
save_summary_tables(summary_tables, TABLES_DIR)

summary_tables["order_status_summary"]

,order_status,transaction_count,percentage
0,Approved,19821,99.1
1,Cancelled,179,0.9


In [6]:
summary_tables["brand_profit_summary"]

,brand,transaction_count,total_profit,average_profit
0,WeareA2B,3295,2753895.17,835.78
1,Solex,4253,2413851.60,567.56
2,Trek Bicycles,2990,1837974.20,614.71
3,Giant Bicycles,3312,1573840.38,475.19
4,OHM Cycles,3043,1483038.84,487.36
5,Norco Bicycles,2910,867683.77,298.17
6,Unknown,197,NaN,NaN


In [7]:
summary_tables["new_customer_state_summary"]

,state,customer_count,total_recent_bike_purchases,avg_recent_bike_purchases
0,NSW,506,25409,50.22
1,VIC,266,12676,47.65
2,QLD,228,11751,51.54


In [8]:
summary_tables["gender_wealth_summary"]

,gender,wealth_segment,customer_count
2,Female,Mass Customer,1044
1,Female,High Net Worth,514
0,Female,Affluent Customer,481
5,Male,Mass Customer,910
4,Male,High Net Worth,482
3,Male,Affluent Customer,481
8,Undisclosed,Mass Customer,46
7,Undisclosed,High Net Worth,25
6,Undisclosed,Affluent Customer,17


## 5. Visualisations

The plots are saved in `reports/figures/` so they can be displayed in the README and used in a portfolio.

In [9]:
create_visualisations(transactions, new_customers, customer_demographic, FIGURES_DIR)
print(f"Figures saved to: {FIGURES_DIR}")

Figures saved to: /mnt/data/kpmg_clean/kpmg-sprocket-customer-analysis/reports/figures


### Order status distribution

![Order status distribution](../reports/figures/order_status_distribution.png)

### Transactions by brand and product size

![Brand by product size](../reports/figures/brand_by_product_size.png)

### New-customer bike purchases by state

![New customer purchases by state](../reports/figures/new_customer_purchases_by_state.png)

### Existing customers by gender and wealth segment

![Gender by wealth segment](../reports/figures/gender_by_wealth_segment.png)

## 6. Key findings

- The order process is strong: most transactions are approved, with only a small number cancelled.
- Solex has the highest transaction count, while WeareA2B produces the highest total profit in the cleaned transaction data.
- Medium-size products dominate across most brands.
- NSW has the strongest new-customer activity by total bike-related purchases in the past three years.
- Mass Customers are the largest wealth segment among both female and male existing customers.

## 7. How this project demonstrates Data Analyst skills

This project shows practical use of Python for importing Excel data, cleaning inconsistent categories, producing summary tables, creating business-focused charts, and communicating findings in a clear and reproducible format.